In [2]:
!nvidia-smi

Sat Sep 26 20:51:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
!rm -f /content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_latest.pt
!rm -f /content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_best.pt
!cd /content/drive/MyDrive/turkish-llm/src && python finetune.py


Cihaz: cuda
Train: 1688 | Val: 188
Base model yuklendi (val loss 2.9573), fine-tuning basliyor.
epoch   1/15 | val loss 0.0647 | 0.1 dk
  -> yeni en iyi checkpoint kaydedildi (/content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_best.pt)
epoch   2/15 | val loss 0.0144 | 0.2 dk
  -> yeni en iyi checkpoint kaydedildi (/content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_best.pt)
epoch   3/15 | val loss 0.0043 | 0.4 dk
  -> yeni en iyi checkpoint kaydedildi (/content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_best.pt)
epoch   4/15 | val loss 0.0028 | 0.5 dk
  -> yeni en iyi checkpoint kaydedildi (/content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_best.pt)
epoch   5/15 | val loss 0.0015 | 0.7 dk
  -> yeni en iyi checkpoint kaydedildi (/content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_best.pt)
epoch   6/15 | val loss 0.0011 | 0.8 dk
  -> yeni en iyi checkpoint kaydedildi (/content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_b

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [37]:
import json
import sys
sys.path.insert(0, "/content/drive/MyDrive/turkish-llm/src")

import torch
from tokenizers import Tokenizer
from model import GPT

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = Tokenizer.from_file("/content/drive/MyDrive/turkish-llm/tokenizer/tokenizer.json")
checkpoint = torch.load("/content/drive/MyDrive/turkish-llm/checkpoints/model_finetuned_best.pt", map_location=DEVICE, weights_only=False)

config = checkpoint["config"]
model = GPT(config).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

eos_id = tokenizer.token_to_id("<eos>")

test_prompts = [
    "Boğaziçi Mühendislik'in bakiyesi ne kadar",
    "Sardunya Tekstil firmasının şu anki durumu ne, borcu var mı",
    "Boğaziçi Mühendislik'e ne kadar param var alacak olarak",
    "Sardunya Tekstil ile geçmişte ne kadar iş yapmışız",
    "FTR-2026-9999 numaralı faturayı getir",
    "Depoda hangi ürünler bitmek üzere, acil ilgilenmemiz lazım",
    "Bugün moralim bozuk, ne önerirsin",
    "en çok satan ürün hangisi",
]

for prompt_text in test_prompts:
    prompt = f"Soru: {prompt_text}\nCevap: "
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long, device=DEVICE)

    output_ids = model.generate(input_ids, max_new_tokens=60, temperature=0.3, top_k=20)
    generated = output_ids[0, input_ids.shape[1]:].tolist()

    if eos_id in generated:
        generated = generated[:generated.index(eos_id)]

    raw_output = tokenizer.decode(generated)

    print("=" * 70)
    print("Soru:", prompt_text)
    print("Ham cikti:", raw_output)
    try:
        parsed = json.loads(raw_output)
        print("Parse edildi:", parsed)
    except Exception as e:
        print("JSON parse edilemedi:", e)


Soru: Boğaziçi Mühendislik'in bakiyesi ne kadar
Ham cikti: {"tool": "get_customer_balance", "arguments": {"customer_name": " Boğaziçi Mühendislik"}}
Parse edildi: {'tool': 'get_customer_balance', 'arguments': {'customer_name': ' Boğaziçi Mühendislik'}}
Soru: Sardunya Tekstil firmasının şu anki durumu ne, borcu var mı
Ham cikti: {"tool": "get_customer_balance", "arguments": {"customer_name": "Sardunya Tekstil"}}
Parse edildi: {'tool': 'get_customer_balance', 'arguments': {'customer_name': 'Sardunya Tekstil'}}
Soru: Boğaziçi Mühendislik'e ne kadar param var alacak olarak
Ham cikti: {"tool": "get_customer_balance", "arguments": {"customer_name": " Boğaziçi Mühendislik"}}
Parse edildi: {'tool': 'get_customer_balance', 'arguments': {'customer_name': ' Boğaziçi Mühendislik'}}
Soru: Sardunya Tekstil ile geçmişte ne kadar iş yapmışız
Ham cikti: {"tool": "get_customer_sales", "arguments": {"customer_name": "Sardunya Tekstil"}}
Parse edildi: {'tool': 'get_customer_sales', 'arguments': {'customer